📝 [최종 모의고사 2차] 작업형 제1유형
문제 1. (시계열 차이 & 그룹핑 - 난이도 상) Machine_ID가 **'M_01'**인 데이터만 추출합니다. 데이터를 시간(Time) 순서대로 정렬한 후, 직전 시간 대비 Temperature(온도)의 **변동폭(절댓값 차이)**을 계산합니다. 이 변동폭이 가장 컸던 시간(Time)의 **'월(Month)'**을 구하시오. (단, 정수로 출력하시오.)

문제 2. (문자열 포함 여부 & 조건부 집계) Status 컬럼에 **'Error'**라는 단어가 포함된 데이터만 필터링합니다. 필터링된 데이터 중에서, Vibration(진동) 컬럼의 결측치를 해당 데이터들의(필터링된 데이터의) 평균값으로 채웁니다. 그 후, 보정된 Vibration 값이 7 이상인 데이터의 개수를 구하시오.

문제 3. (Min-Max Scaling & 시간 조건) Time 컬럼에서 **'오전(6시~11시, 6<=x<=11)'**에 해당하는 데이터만 추출합니다. 이 데이터의 Power_Usage 컬럼을 Min-Max Scaling을 적용하여 변환합니다. (sklearn 미사용 권장) 변환된 값이 0.5 보다 큰(> 0.5) 데이터들의 원래 Power_Usage 평균값을 구하시오. (단, 소수점 이하는 버리고 **정수(int)**로 출력하시오.)

In [16]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n_rows = 3000

# 1. 데이터 생성
times = pd.date_range('2023-01-01', periods=n_rows, freq='h')
machines = np.random.choice(['M_01', 'M_02', 'M_03', 'M_04', 'M_05'], n_rows)

# 상태 코드 생성 (정상, 경고, 에러)
status_opts = ['[Normal]', '[Warning: Overheat]', '[Warning: Noise]', '[Error: 001]', '[Error: 002]']
status_list = np.random.choice(status_opts, n_rows, p=[0.8, 0.1, 0.05, 0.03, 0.02])

df = pd.DataFrame({
    'Time': times,
    'Machine_ID': machines,
    'Temperature': np.random.normal(60, 10, n_rows),
    'Vibration': np.random.normal(5, 2, n_rows),
    'Power_Usage': np.random.randint(100, 500, n_rows),
    'Status': status_list
})

# 결측치 주입 (Vibration)
df.loc[np.random.choice(df.index, 50), 'Vibration'] = np.nan

# 날짜 정렬 (뒤죽박죽 섞음 -> 문제에서 정렬 유도)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("최종 모의고사 2차 데이터 생성 완료!")
print(df.head())

최종 모의고사 2차 데이터 생성 완료!
                 Time Machine_ID  Temperature  Vibration  Power_Usage  \
0 2023-03-17 01:00:00       M_04    47.091690   4.449014          187   
1 2023-02-19 14:00:00       M_02    70.419261   5.595863          221   
2 2023-03-17 17:00:00       M_01    69.790871   5.424813          247   
3 2023-01-11 11:00:00       M_05    53.821372   4.643010          401   
4 2023-04-15 09:00:00       M_04    41.590929   4.636016          300   

             Status  
0          [Normal]  
1          [Normal]  
2  [Warning: Noise]  
3          [Normal]  
4          [Normal]  


In [36]:
# 1번 문제
df['Time'] = pd.to_datetime(df['Time'])
df_1 = df[df['Machine_ID']=='M_01']
df_1 = df_1.sort_values(by='Time', ascending=False)
print(df_1.head())

df_1['Diff'] = df_1['Temperature'].diff().abs()
idx = df_1['Diff'].idxmax()

result = df.loc[idx]
print(result)
# 답: 2월

                    Time Machine_ID  Temperature  Vibration  Power_Usage  \
399  2023-05-05 20:00:00       M_01    81.391611   2.350236          145   
2455 2023-05-05 16:00:00       M_01    67.835137   8.010379          210   
2296 2023-05-05 08:00:00       M_01    68.910092   6.160863          125   
1967 2023-05-04 19:00:00       M_01    64.416013   6.907532          298   
120  2023-05-04 15:00:00       M_01    76.623780   7.351757          256   

                   Status  hours  
399              [Normal]     20  
2455         [Error: 002]     16  
2296             [Normal]      8  
1967             [Normal]     19  
120   [Warning: Overheat]     15  
Time           2023-02-09 01:00:00
Machine_ID                    M_01
Temperature              87.456967
Vibration                 6.612338
Power_Usage                    310
Status         [Warning: Overheat]
hours                            1
Name: 943, dtype: object


In [ ]:
# 2번 문제
df_2 = df[df['Status'].str.contains('Error')].copy()
mean_error = df_2['Vibration'].mean()
df_2['Vibration'] = df_2['Vibration'].fillna(mean_error)

result = len(df_2[df_2['Vibration'] >= 7])
print(result)
# 답: 20

20


In [ ]:
# 3번 문제
df['Time'] = pd.to_datetime(df['Time'])
df['hours'] = df['Time'].dt.hour
q1 = df[(df['hours'] >= 6) & (df['hours'] <= 11)].copy()
#print(q1)

min_p = q1['Power_Usage'].min()
max_p = q1['Power_Usage'].max()

q1['Scaled'] = (q1['Power_Usage'] - min_p) / (max_p - min_p)
mean_s = q1[q1['Scaled'] > 0.5]['Power_Usage'].mean()
print(int(mean_s))
# 답: 397

397


📝 [제2유형] 여행자 보험 가입 예측 (Simple Ver.)
문제: 제공된 학습 데이터(train)를 이용하여 고객의 보험 가입(청구) 여부(Claim)를 예측하는 모델을 만들고, 평가용 데이터(test)에 대한 예측 결과를 result.csv 파일로 제출하시오.

1. 데이터 설명

Target: Claim (1: 가입, 0: 미가입)

Features:

Age: 나이

Agency, Agency Type: 범주형 변수 (인코딩 필요)

Commision, Duration: 수치형 변수 (Duration에 결측치 있음)

2. 제출 형식

result.csv 파일로 저장.

**ID**와 pred 두 개의 컬럼만 포함.

pred 컬럼에는 **가입할 확률(Probability)**을 제출하시오. (0~1 사이 실수)

3. 평가 지표

ROC-AUC Score

In [48]:
import pandas as pd
import numpy as np

# 랜덤 시드 고정
np.random.seed(2025)
n_rows = 1500

# 1. 데이터 생성 (순수 numpy 사용)
data = {
    'ID': range(1001, 1001 + n_rows),
    'Age': np.random.randint(20, 80, n_rows),
    'Agency': np.random.choice(['Agency_A', 'Agency_B', 'Agency_C'], n_rows),
    'Agency Type': np.random.choice(['Airlines', 'Travel Agency'], n_rows),
    'Commision': np.random.uniform(0, 200, n_rows),
    'Duration': np.random.randint(1, 100, n_rows),
    'Claim': 0 # 초기화
}

df = pd.DataFrame(data)

# 2. Target 생성 (나이 많고, 수수료 높을수록 가입 확률 높음)
# 간단한 로직: 점수 > 평균이면 1, 아니면 0 (노이즈 추가)
score = df['Age'] * 0.5 + df['Commision'] * 2 + np.random.normal(0, 20, n_rows)
threshold = score.median()
df['Claim'] = np.where(score > threshold, 1, 0)

# 3. 결측치 주입 (Duration)
df.loc[np.random.choice(df.index, 50), 'Duration'] = np.nan

# 4. Train / Test 분리 (수동 분리)
train = df.iloc[:1000]
test = df.iloc[1000:].drop('Claim', axis=1) # Test에는 타겟 제거

print("데이터 생성 완료!")
print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(train.head())

데이터 생성 완료!
Train shape: (1000, 7), Test shape: (500, 6)
     ID  Age    Agency    Agency Type   Commision  Duration  Claim
0  1001   50  Agency_B  Travel Agency  147.538421      75.0      1
1  1002   38  Agency_B  Travel Agency   57.659979      50.0      0
2  1003   50  Agency_B       Airlines  135.676047      94.0      1
3  1004   32  Agency_A       Airlines  110.596363       NaN      1
4  1005   76  Agency_C  Travel Agency   45.992067      67.0      0


In [52]:
#print(test.info())
# 인코딩: Agency, Agency Type
# 결측치: Duration

mean_d = train['Duration'].mean()
train['Duration'] = train['Duration'].fillna(mean_d)
test['Duration'] = test['Duration'].fillna(mean_d)

X = train.drop(['ID', 'Claim'], axis=1)
y = train['Claim']
X_submit = test.drop(['ID'], axis=1)

cols = ['Agency', 'Agency Type']
ct = pd.concat([X, X_submit])
ct = pd.get_dummies(ct, columns=cols)
X = ct.iloc[:len(X)]
X_submit = ct.iloc[len(X):]

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
pred_proba = model.predict_proba(X_val)[:,1]

from sklearn.metrics import roc_auc_score
score = roc_auc_score(y_val, pred_proba)
print(round(score, 3))

model = RandomForestClassifier(random_state=42)
model.fit(X, y)
pred = model.predict_proba(X_submit)[:,1]

result = pd.DataFrame({
    'ID': test['ID'],
    'pred': pred
})

#result.to_csv('result.csv', index=False)
print(result.head())

C:\Users\sangh\AppData\Local\Temp\ipykernel_13052\2200849869.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['Duration'] = train['Duration'].fillna(mean_d)


0.986
        ID  pred
1000  2001  0.97
1001  2002  0.97
1002  2003  0.12
1003  2004  0.00
1004  2005  0.98


📝 [제3유형] 회귀분석 심화
문제 1. (다중선형회귀 - 회귀계수) 직원의 성과 점수(Performance)를 종속변수로 하고, Years_Exp(경력), Salary(연봉), Job_Sat(만족도)를 독립변수로 하는 다중선형회귀(OLS) 모델을 학습시키시오. 이때, **Years_Exp 변수의 회귀계수(Coefficient)**를 구하시오. (단, statsmodels를 사용하며, 결과는 반올림하여 소수 셋째 자리까지 출력하시오.)

문제 2. (로지스틱 회귀 - 오즈비) 직원의 퇴사 여부(Attrition)를 종속변수로 하고, Age, Salary, Job_Sat을 독립변수로 하는 로지스틱 회귀(Logit) 모델을 학습시키시오. 이때, Job_Sat(직무 만족도)가 1단위 증가할 때, 퇴사할 오즈(Odds)는 몇 배가 되는지(오즈비) 구하시오. (단, 결과는 반올림하여 소수 넷째 자리까지 출력하시오.)

문제 3. (상관분석 - 통계량) Salary(연봉)와 Performance(성과 점수) 간의 **피어슨 상관계수(Pearson Correlation Coefficient)**를 구하시오. (단, scipy.stats를 사용하며, 결과는 반올림하여 소수 셋째 자리까지 출력하시오.)

In [53]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n = 400

# 1. 독립변수 생성
data = {
    'ID': range(1001, 1001 + n),
    'Age': np.random.randint(24, 60, n),
    'Years_Exp': np.random.randint(1, 20, n),
    'Job_Sat': np.random.randint(1, 6, n)
}
df = pd.DataFrame(data)

# Salary: 경력과 나이에 비례
df['Salary'] = 30000 + (df['Years_Exp'] * 2000) + (df['Age'] * 500) + np.random.normal(0, 5000, n)

# 2. 종속변수 1 (Performance): 경력과 만족도가 높을수록 높음
# 식: 20 + 2*Exp + 5*Sat + Error
df['Performance'] = 20 + (df['Years_Exp'] * 2.0) + (df['Job_Sat'] * 5.0) + np.random.normal(0, 5, n)
df['Performance'] = df['Performance'].clip(0, 100)

# 3. 종속변수 2 (Attrition): 만족도가 낮고 연봉이 낮을수록 퇴사 확률 높음
# 식: Logit = 2 - 0.8*Sat - 0.00005*Salary
logits = 2 - (0.8 * df['Job_Sat']) - (0.00005 * df['Salary'])
probs = 1 / (1 + np.exp(-logits))
df['Attrition'] = np.random.binomial(1, probs)

print("데이터 생성 완료!")
print(df.head())

데이터 생성 완료!
     ID  Age  Years_Exp  Job_Sat         Salary  Performance  Attrition
0  1001   54         10        1   82074.907623    41.559047          0
1  1002   42         19        3   90736.044002    67.836166          0
2  1003   54         19        1  102151.239802    70.295290          0
3  1004   36         18        3   86337.519682    78.821604          0
4  1005   27          1        4   43705.411846    39.830261          0


In [56]:
from statsmodels.formula.api import ols
model1 = ols('Performance ~ Years_Exp + Salary + Job_Sat', data=df).fit()
coef_1 = model1.params['Years_Exp']
print(round(coef_1, 3))

from statsmodels.formula.api import logit
import numpy as np
model2 = logit('Attrition ~ Age + Salary + Job_Sat', data=df).fit()
odds_ratio = np.exp(model2.params['Job_Sat'])
print(round(odds_ratio, 4))

from scipy.stats import pearsonr
stat, p_val = pearsonr(df['Salary'], df['Performance'])
print(round(stat, 3))

1.972
Optimization terminated successfully.
         Current function value: 0.088568
         Iterations 9
0.4478
0.67


📝 [제1유형 기초] loc vs iloc 연습
문제 1. (행 선택)

loc 사용: 학생 이름이 **'민수'**인 행 전체를 출력하시오.

iloc 사용: 위와 똑같은 행을 **순서 번호(인덱스 번호)**를 사용하여 출력하시오. ('민수'는 위에서 3번째입니다.)

문제 2. (슬라이싱 - 범위 선택)

loc 사용: '영희'부터 '지수'까지의 국어, 영어 점수를 출력하시오.

(주의: loc 슬라이싱은 끝을 포함할까요?)

iloc 사용: 위와 똑같은 범위를 순서 번호로 슬라이싱하여 출력하시오.

(주의: iloc 슬라이싱은 끝 번호를 포함할까요?)

문제 3. (조건 필터링 - loc 전용)

loc 사용: 수학 점수가 80점 이상인 학생들의 **'영어'**와 '과학' 점수만 출력하시오.

(힌트: df.loc[조건, [컬럼명들]])

문제 4. (값 수정)

iloc 사용: **마지막 학생(동현)**의 마지막 과목(과학) 점수를 0점으로 수정하시오. (이름을 쓰지 말고 위치 번호로만 접근하세요.)

In [57]:
import pandas as pd

data = {
    '국어': [90, 80, 70, 100, 50],
    '영어': [85, 95, 75, 80, 60],
    '수학': [100, 90, 80, 70, 50],
    '과학': [88, 77, 66, 55, 44]
}
names = ['철수', '영희', '민수', '지수', '동현']

# '이름'을 인덱스로 설정 (loc 실습을 위해 필수!)
df = pd.DataFrame(data, index=names)

print("데이터 생성 완료!")
print(df)

데이터 생성 완료!
     국어  영어   수학  과학
철수   90  85  100  88
영희   80  95   90  77
민수   70  75   80  66
지수  100  80   70  55
동현   50  60   50  44


In [72]:
result1 = df.loc['민수']
#print(result1)

result2 = df.iloc[2]
#print(result2)

result3 = df.loc['영희':'지수']
#print(result3)

result4 = df.iloc[1:4]
#print(result4)

result5 = df.loc[df['수학']>=80, ['영어', '과학']]
print(result5)

    영어  과학
철수  85  88
영희  95  77
민수  75  66


📝 [제3유형] 로지스틱 회귀 & 확률 예측
문제 1. (오즈비) Churn을 종속변수로, Age, Monthly_Bill, Contract를 독립변수로 하는 로지스틱 회귀 모델을 학습시키시오. 이때, Monthly_Bill (월 요금)이 1단위 증가할 때 이탈할 오즈(Odds)는 몇 배가 되는지 구하시오. (단, statsmodels를 사용하며, 반올림하여 소수 셋째 자리까지 출력하시오.)

문제 2. (확률 예측 & 조건부 개수) 위 1번에서 만든 모델을 사용하여 전체 고객의 **이탈 확률(Probability)**을 예측하시오. 예측된 이탈 확률이 **0.6 이상(>= 0.6)**인 고객은 총 몇 명인지 구하시오. (단, 정수(int)로 출력하시오.)

문제 3. (오분류율) 위 모델의 예측 확률을 기반으로, 확률이 **0.5 이상이면 1(이탈), 미만이면 0(유지)**으로 분류합니다. 이 예측값과 실제값(Churn)을 비교하여 **오분류율(Misclassification Rate)**을 구하시오. (단, 반올림하여 소수 둘째 자리까지 출력하시오.)

In [73]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n = 400

data = {
    'ID': range(1001, 1001 + n),
    'Age': np.random.randint(20, 70, n),
    'Monthly_Bill': np.random.normal(60, 15, n),
    'Contract': np.random.choice(['Month', 'Year'], n, p=[0.6, 0.4]),
    'Churn': 0
}
df = pd.DataFrame(data)

# 이탈 확률 생성 (월 요금이 비싸고, 약정이 짧을수록(Month) 이탈 확률 높음)
is_month = np.where(df['Contract'] == 'Month', 1, 0)
logits = -4 + (df['Monthly_Bill'] * 0.05) + (is_month * 1.5) - (df['Age'] * 0.01)
probs = 1 / (1 + np.exp(-logits))
df['Churn'] = np.random.binomial(1, probs)

print("데이터 생성 완료!")
print(df.head())

데이터 생성 완료!
     ID  Age  Monthly_Bill Contract  Churn
0  1001   50     57.398437     Year      0
1  1002   38     52.444274     Year      0
2  1003   50     65.285320    Month      1
3  1004   32     69.985507    Month      1
4  1005   23     76.520891    Month      1


In [79]:
from statsmodels.formula.api import logit
import numpy as np
model = logit('Churn ~ Age + Monthly_Bill + C(Contract)', data=df).fit()
odds_ratio = np.exp(model.params['Monthly_Bill'])
print(round(odds_ratio, 3))

pred = model.predict(df)
target = pred[pred>=0.6]
count = len(target)
print(count)

from sklearn.metrics import accuracy_score
pred_class = np.where(pred>=0.5, 1, 0)
score = accuracy_score(df['Churn'], pred_class)
mis_rate = 1 - score
print(round(mis_rate, 2))

Optimization terminated successfully.
         Current function value: 0.571011
         Iterations 6
1.045
78
0.33


📝 [제3유형] Z-검정, F-검정, ANOVA
문제 1. (2표본 Z-검정 - 비율 검정) 'Control' 그룹과 'Test_A' 그룹 간의 **구매 전환율(Converted 비율)**에 차이가 있는지 검정하려고 합니다. **두 집단의 비율 차이에 대한 Z-검정(Two-proportions Z-test)**을 수행하고, **p-값(p-value)**을 구하시오. (단, 양측 검정(two-sided)을 수행하며, 결과는 반올림하여 소수 넷째 자리까지 출력하시오.)

문제 2. (F-검정 - 등분산 검정) 'Control' 그룹과 'Test_A' 그룹의 Time_Spent(체류 시간) 데이터가 **등분산(Equal Variance)**을 만족하는지 확인하기 위해 F-검정을 수행하려고 합니다. 두 그룹의 분산을 각각 구한 뒤, **F-검정 통계량(F-ratio)**을 구하시오. (단, F-통계량은 큰 분산 / 작은 분산으로 계산하며, 분산 계산 시 ddof=1을 적용하고, 결과는 소수 셋째 자리까지 출력하시오.)

문제 3. (일원분산분석 - ANOVA) 세 그룹(Control, Test_A, Test_B) 간의 평균 Time_Spent에 차이가 있는지 검정하기 위해 ANOVA를 수행하시오. 이때의 F-통계량을 구하시오. (단, 결과는 반올림하여 소수 셋째 자리까지 출력하시오.)

In [80]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n = 300

# 데이터 생성
groups = ['Control'] * 100 + ['Test_A'] * 100 + ['Test_B'] * 100
data = {
    'ID': range(1, n + 1),
    'Group': groups,
    'Time_Spent': np.concatenate([
        np.random.normal(120, 20, 100),  # Control
        np.random.normal(130, 25, 100),  # Test_A (분산 큼)
        np.random.normal(135, 20, 100)   # Test_B
    ]),
    'Converted': np.concatenate([
        np.random.binomial(1, 0.2, 100), # Control (20%)
        np.random.binomial(1, 0.35, 100),# Test_A (35%)
        np.random.binomial(1, 0.25, 100) # Test_B (25%)
    ])
}

df = pd.DataFrame(data)

print("데이터 생성 완료!")
print(df.groupby('Group')[['Time_Spent', 'Converted']].mean())

데이터 생성 완료!
         Time_Spent  Converted
Group                         
Control  117.002655       0.19
Test_A   131.868632       0.37
Test_B   134.126698       0.20


In [90]:
from statsmodels.stats.proportion import proportions_ztest
print(df.head())

df_c = len(df[df['Group']=='Control'])
df_a = len(df[df['Group']=='Test_A'])
len_c = len(df[(df['Group']=='Control') & (df['Converted']==1)])
len_a = len(df[(df['Group']=='Test_A') & (df['Converted']==1)])

stat, p_val = proportions_ztest(count=[len_c, len_a], nobs=[df_c, df_a])
print(round(p_val, 4))

import numpy as np
var1 = np.var(df[(df['Group']=='Control')]['Time_Spent'], ddof=1)
var2 = np.var(df[(df['Group']=='Test_A')]['Time_Spent'], ddof=1)

#print(var1)
#print(var2)

f_stat = var2 / var1
print(round(f_stat, 3))

from scipy.stats import f_oneway
g1 = df[df['Group']=='Control']['Time_Spent']
g2 = df[df['Group']=='Test_A']['Time_Spent']
g3 = df[df['Group']=='Test_B']['Time_Spent']
stat, p_val = f_oneway(g1, g2, g3)
print(round(stat, 3))

   ID    Group  Time_Spent  Converted
0   1  Control  118.152196          1
1   2  Control  134.685711          1
2   3  Control   91.222359          0
3   4  Control  106.731560          0
4   5  Control  117.985438          0
0.0046
1.709
19.866


In [96]:
import statsmodels
print(dir(statsmodels))
import pandas as pd

# 함수 자체를 넣으세요 (괄호() 빼고!)
print(help(pd.read_csv))

# 출력된 설명에서 'encoding' 파라미터 설명을 찾으면 됨

['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '__version_info__', '__version_tuple__', '_version', 'base', 'compat', 'debug_warnings', 'discrete', 'distributions', 'duration', 'emplike', 'formula', 'gam', 'genmod', 'graphics', 'iolib', 'monkey_patch_cat_dtype', 'regression', 'robust', 'stats', 'test', 'tools']
Help on function read_csv in module pandas.io.parsers.readers:

read_csv(filepath_or_buffer: 'FilePath | ReadCsvBuffer[bytes] | ReadCsvBuffer[str]', *, sep: 'str | None | lib.NoDefault' = <no_default>, delimiter: 'str | None | lib.NoDefault' = None, header: "int | Sequence[int] | None | Literal['infer']" = 'infer', names: 'Sequence[Hashable] | None | lib.NoDefault' = <no_default>, index_col: 'IndexLabel | Literal[False] | None' = None, usecols: 'UsecolsArgType' = None, dtype: 'DtypeArg | None' = None, engine: 'CSVEngine | None' = None, converters: 'Mapping[Hashable, Callable] | None

📝 [제3유형] 교육 방법별 점수 차이 분석
문제 1. (등분산 검정 - Levene) 두 교육 방법(Online, Offline) 간의 점수(Score) 분산이 같은지 검정하고자 합니다. Levene의 등분산 검정을 수행하고, **p-값(p-value)**을 구하시오. (단, 반올림하여 소수 셋째 자리까지 출력하시오.)

문제 2. (독립표본 t-검정) 위 1번의 결과(p-값)를 바탕으로 등분산 가정 여부를 결정하고, 두 그룹 간의 평균 점수에 유의미한 차이가 있는지 독립표본 t-검정을 수행하시오. 이때의 **t-검정 통계량(t-statistic)**을 구하시오. (절댓값이 아닌 부호 있는 원본 값) (단, 반올림하여 소수 셋째 자리까지 출력하시오.)

In [97]:
import pandas as pd
import numpy as np

np.random.seed(2025)
n = 60

# 데이터 생성
# Online: 평균 75, 분산 100 (표준편차 10)
group_online = np.random.normal(75, 10, 30)
# Offline: 평균 82, 분산 144 (표준편차 12)
group_offline = np.random.normal(82, 12, 30)

df = pd.DataFrame({
    'ID': range(1, n + 1),
    'Method': ['Online'] * 30 + ['Offline'] * 30,
    'Score': np.concatenate([group_online, group_offline])
})

print("데이터 생성 완료!")
print(df.groupby('Method')['Score'].describe())

데이터 생성 완료!
         count       mean        std        min        25%        50%  \
Method                                                                  
Offline   30.0  81.207741  11.682591  65.281171  70.601313  80.402981   
Online    30.0  75.905395   8.828388  57.106770  72.972759  74.688137   

               75%         max  
Method                          
Offline  91.964102  104.818489  
Online   82.301892   96.469866  


In [ ]:
#print(df.head())

from scipy.stats import levene, ttest_ind
on = df[df['Method']=='Online']['Score']
off = df[df['Method']=='Offline']['Score']

stat, p_val = levene(on, off)
print(round(p_val, 3))
# 출력: 0.023 (0.05보다 작으므로 '이분산'이다!)

# 등분산이 아니므로(이분산) equal_var=False로 설정해야 합니다.
stat, p_val = ttest_ind(on, off, equal_var=False)
print(round(stat, 3))

0.023
-1.983
